In [2]:
import zipfile
import numpy as np
import pandas as pd

zip_path = '../Data_Folder/Large_AML_CSV.zip'

features_list = []
subject_ids = []

with zipfile.ZipFile(zip_path, 'r') as z:
    all_csvs = sorted([f for f in z.namelist() if f.endswith('.CSV')])
    total_subjects = len(all_csvs) // 8
    
    for subject_idx in range(total_subjects):
        subject_features = []
        subject_tubes = all_csvs[subject_idx * 8 : (subject_idx + 1) * 8]
        
        for tube_file in subject_tubes:
            with z.open(tube_file) as f:
                df = pd.read_csv(f)
                
                for col in df.columns:
                    vals = df[col].values
                    subject_features.extend([
                        np.mean(vals),
                        np.std(vals),
                        np.median(vals),
                        np.percentile(vals, 90),
                        np.percentile(vals, 10)
                    ])
                    
        features_list.append(subject_features)
        subject_ids.append(subject_idx + 1)
        
        if (subject_idx + 1) % 20 == 0 or (subject_idx + 1) == total_subjects:
            print(f"Processed {subject_idx + 1}/{total_subjects} subjects...")

X_all = pd.DataFrame(features_list)
print("Extraction complete! Matrix shape:", X_all.shape)

Processed 20/359 subjects...
Processed 40/359 subjects...
Processed 60/359 subjects...
Processed 80/359 subjects...
Processed 100/359 subjects...
Processed 120/359 subjects...
Processed 140/359 subjects...
Processed 160/359 subjects...
Processed 180/359 subjects...
Processed 200/359 subjects...
Processed 220/359 subjects...
Processed 240/359 subjects...
Processed 260/359 subjects...
Processed 280/359 subjects...
Processed 300/359 subjects...
Processed 320/359 subjects...
Processed 340/359 subjects...
Processed 359/359 subjects...
Extraction complete! Matrix shape: (359, 280)


In [6]:
display (X_all)

,0,1,2,3,4,5,6,7,8,9,...,270,271,272,273,274,275,276,277,278,279
0,663.982311,218.845701,713.0,934.0,387.0,0.554379,0.095015,0.602538,0.648491,0.422839,...,0.168969,0.051880,0.156796,0.197424,0.140279,0.160188,0.029755,0.153350,0.185781,0.140279
1,567.782309,248.536705,615.0,884.0,247.4,0.537679,0.089433,0.576743,0.628206,0.400863,...,0.172091,0.044139,0.164940,0.202304,0.140279,0.161918,0.028706,0.157195,0.187011,0.140279
2,605.216605,194.071950,636.0,836.0,309.0,0.581185,0.081645,0.608744,0.655534,0.439851,...,0.199506,0.100475,0.172269,0.238034,0.141874,0.173292,0.059372,0.159655,0.197424,0.140279
3,609.055791,219.358016,665.0,857.0,260.0,0.573792,0.094242,0.613173,0.659053,0.417132,...,0.173082,0.053062,0.162236,0.204454,0.140279,0.164838,0.034715,0.159237,0.192088,0.140279
4,666.791300,193.052973,679.0,924.0,409.0,0.503988,0.076395,0.505559,0.601651,0.403745,...,0.213511,0.059141,0.206274,0.281432,0.145623,0.161466,0.024194,0.156796,0.188256,0.140279
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354,547.962232,206.101993,592.0,789.0,234.0,0.538303,0.095291,0.578527,0.628206,0.377645,...,0.235284,0.086892,0.213405,0.330493,0.152983,0.168347,0.038875,0.159655,0.199840,0.140279
355,696.730600,179.319717,726.0,899.0,394.0,0.571268,0.072063,0.594547,0.628206,0.436084,...,0.185783,0.041122,0.180448,0.219672,0.148801,0.155068,0.020725,0.150149,0.175436,0.140279
356,423.905917,207.767709,387.0,787.0,199.0,0.453769,0.098565,0.420939,0.613173,0.353178,...,0.229049,0.123811,0.171754,0.417703,0.140279,0.168402,0.059400,0.140802,0.234581,0.140279
357,548.570323,188.576547,558.0,795.0,322.0,0.554542,0.092435,0.603425,0.648491,0.425685,...,0.165086,0.038498,0.157597,0.192741,0.140279,0.157124,0.022788,0.152257,0.179876,0.140279


In [7]:
import pandas as pd

# Load labels
labels_df = pd.read_csv('../Data_Folder/Class_Labels.csv')

# Option A: If Class_Labels.csv has 2,872 rows (1 per tube file),
# take 1 label per subject (every 8th row) for the 179 training subjects:
y_train_raw = labels_df['Label'].iloc[: 179 * 8 : 8]
y_train = y_train_raw.map({'normal': 0, 'AML': 1}).values

# Print updated shapes to confirm alignment
print("Fixed X_train shape:", X_train.shape)
print("Fixed y_train shape:", y_train.shape)

Fixed X_train shape: (179, 280)
Fixed y_train shape: (179,)


In [11]:
# 1. Extract 1 label per subject (every 8th row)
y_train_raw = labels_df['Label'].iloc[: 179 * 8 : 8]

# 2. Map lowercase 'normal' -> 0 and 'aml' -> 1
y_train = y_train_raw.str.strip().map({'normal': 0, 'aml': 1}).values

# 3. Verify target shape and NaN check
print("Any NaN in y_train?", np.isnan(y_train).any())
print("Class breakdown (0=normal, 1=aml):", pd.Series(y_train).value_counts().to_dict())

Any NaN in y_train? False
Class breakdown (0=normal, 1=aml): {0: 156, 1: 23}


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# 1. Scale features using training set metrics
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Initialize Logistic Regression with class weighting
model = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0)

# 3. Compute 5-Fold Cross-Validation ROC-AUC
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')

print(f"5-Fold CV ROC-AUC Score: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

5-Fold CV ROC-AUC Score: 0.9948 +/- 0.0103


In [14]:
# 1. Fit the final model on all 179 scaled training subjects
model.fit(X_train_scaled, y_train)

# 2. Predict probabilities for the 180 unlabeled test subjects
# Column 1 represents the probability of being AML (class 1)
test_probabilities = model.predict_proba(X_test_scaled)[:, 1]

# 3. Convert probabilities to binary predictions (threshold = 0.5)
test_predictions = (test_probabilities >= 0.5).astype(int)

# --- Summary Statistics & Percentage Calculations ---

# Ground Truth Counts
train_aml_actual = sum(y_train == 1)      # 23
train_normal_actual = sum(y_train == 0)   # 156
total_train = len(y_train)                # 179

test_aml_actual = 20                      # Ground truth based on total 43 AML cases
test_normal_actual = 160                  # 180 total - 20 actual AML
total_test = len(test_predictions)        # 180

# Predicted Counts (Test Set)
test_aml_pred = sum(test_predictions == 1)    # 19
test_normal_pred = sum(test_predictions == 0) # 161

# Output Display
print("=" * 55)
print("               TRAINING SET BREAKDOWN (ACTUAL)         ")
print("=" * 55)
print(f"Actual Normal Cases: {train_normal_actual:3d} | ({train_normal_actual / total_train * 100:.2f}%)")
print(f"Actual AML Cases:    {train_aml_actual:3d} | ({train_aml_actual / total_train * 100:.2f}%)")
print(f"Total Training:      {total_train:3d} | (100.00%)")

print("\n" + "=" * 55)
print("               TEST SET BREAKDOWN (ACTUAL vs PREDICTED) ")
print("=" * 55)
print(f"Actual Normal Cases:    {test_normal_actual:3d} | ({test_normal_actual / total_test * 100:.2f}%)")
print(f"Predicted Normal Cases: {test_normal_pred:3d} | ({test_normal_pred / total_test * 100:.2f}%)")
print("-" * 55)
print(f"Actual AML Cases:       {test_aml_actual:3d} | ({test_aml_actual / total_test * 100:.2f}%)")
print(f"Predicted AML Cases:    {test_aml_pred:3d} | ({test_aml_pred / total_test * 100:.2f}%)")
print("-" * 55)
print(f"Total Test Cohort:      {total_test:3d} | (100.00%)")
print("=" * 55)

               TRAINING SET BREAKDOWN (ACTUAL)         
Actual Normal Cases: 156 | (87.15%)
Actual AML Cases:     23 | (12.85%)
Total Training:      179 | (100.00%)

               TEST SET BREAKDOWN (ACTUAL vs PREDICTED) 
Actual Normal Cases:    160 | (88.89%)
Predicted Normal Cases: 161 | (89.44%)
-------------------------------------------------------
Actual AML Cases:        20 | (11.11%)
Predicted AML Cases:     19 | (10.56%)
-------------------------------------------------------
Total Test Cohort:      180 | (100.00%)


In [17]:
import pandas as pd

# Create a clear results DataFrame
results_df = pd.DataFrame({
    'Subject_Index': range(179, 359),
    'Predicted_Label': test_predictions,
    'AML_Probability': test_probabilities
})

# Map 0 and 1 back to string labels if required
results_df['Predicted_Class'] = results_df['Predicted_Label'].map({0: 'normal', 1: 'aml'})

# Save to CSV in your project folder
results_df.to_csv('../Data_Folder/Test_Predictions.csv', index=False)

print("Saved test predictions to '../Data_Folder/Test_Predictions.csv'")
results_df.head()

Saved test predictions to '../Data_Folder/Test_Predictions.csv'


,Subject_Index,Predicted_Label,AML_Probability,Predicted_Class
0,179,0,0.000700,normal
1,180,0,0.000985,normal
2,181,0,0.000133,normal
3,182,0,0.000207,normal
4,183,0,0.001483,normal
